# Load Qwen/Qwen3-30B-A3B

Loads the model with SGLang offline engine. MoE expert weights are kept in CPU RAM (`enable_expert_vm`).

**Reduce KV cache GPU memory:**
- `mem_fraction_static` — lower fraction → smaller KV pool (default was 0.80 → ~72 GB KV)
- `max_total_tokens` — hard cap on cached tokens (e.g. 100_000 ≈ ~9 GB KV at bf16)
- Optional: `kv_cache_dtype="fp8_e4m3"` — ~half KV size if your GPU supports FP8 KV

In [1]:
import nest_asyncio

nest_asyncio.apply()

In [2]:
import sglang as sgl

MODEL_PATH = "Qwen/Qwen3-30B-A3B"
MODEL_PATH = "Qwen/Qwen3-30B-A3B-GPTQ-Int4"

print(f"Loading {MODEL_PATH}...")

llm = sgl.Engine(
    model_path=MODEL_PATH,
    tp_size=1,
    dtype="bfloat16",
    trust_remote_code=True,
    enable_expert_vm=True,
    expert_vm_resident_layers="0",
    disable_cuda_graph=True,
    mem_fraction_static=0.1,
    max_total_tokens=100,
    log_level="info",
)

print("Model loaded successfully.")

Loading Qwen/Qwen3-30B-A3B-GPTQ-Int4...


[2026-06-08 06:31:50] server_args=ServerArgs(model_path='Qwen/Qwen3-30B-A3B-GPTQ-Int4', tokenizer_path='Qwen/Qwen3-30B-A3B-GPTQ-Int4', tokenizer_mode='auto', tokenizer_backend='huggingface', tokenizer_worker_num=1, detokenizer_worker_num=1, skip_tokenizer_init=False, load_format='auto', model_loader_extra_config='{}', trust_remote_code=True, context_length=None, is_embedding=False, prefill_only_disable_kv_cache=False, enable_multimodal=None, revision=None, model_impl='auto', model_config_parser='auto', host='127.0.0.1', port=30000, fastapi_root_path='', grpc_mode=False, skip_server_warmup=False, warmups=None, nccl_port=None, checkpoint_engine_wait_weights_before_ready=False, ssl_keyfile=None, ssl_certfile=None, ssl_ca_certs=None, ssl_keyfile_password=None, enable_ssl_refresh=False, enable_http2=False, dtype='bfloat16', quantization=None, quantization_param_path=None, kv_cache_dtype='auto', enable_fp32_lm_head=False, modelopt_quant=None, modelopt_checkpoint_restore_path=None, modelopt_c

Model loaded successfully.


In [3]:
outputs = llm.generate("Hello", sampling_params={"max_new_tokens": 5})
outputs

Process Process-2:
Process Process-1:
Traceback (most recent call last):
  File "/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/teamspace/studios/this_studio/sglang/python/sglang/srt/managers/scheduler.py", line 3885, in run_scheduler_process
    scheduler.run_event_loop()
  File "/teamspace/studios/this_studio/sglang/python/sglang/srt/managers/scheduler.py", line 1363, in run_event_loop
    dispatch_event_loop(self)
  File "/teamspace/studios/this_studio/sglang/python/sglang/srt/managers/scheduler.py", line 3754, in dispatch_event_loop
    scheduler.event_loop_overlap()
  File "/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/torch/utils/_contextlib.py", line 124, in decorate_context
    return func(*args, **kwargs)
           ^^^^^^^^

KeyboardInterrupt: 

[rank0]:[W608 06:36:51.200401965 ProcessGroupNCCL.cpp:1575] Warning: WARNING: destroy_process_group() was not called before program exit, which can leak resources. For more info, please see https://pytorch.org/docs/stable/distributed.html#shutdown (function operator())
[2026-06-08 06:36:58] Subprocess detokenizer (pid=14494) crashed with exit code 1. Triggering SIGQUIT for cleanup...
[2026-06-08 06:36:58] SIGQUIT received. signum=None, frame=None. It usually means one child failed.


: 

In [3]:
import psutil

mem = psutil.virtual_memory()
print(f"Total:     {mem.total / 2**30:.2f} GiB")
print(f"Available: {mem.available / 2**30:.2f} GiB")
print(f"Used:      {mem.used / 2**30:.2f} GiB")
print(f"Percent:   {mem.percent}%")

Total:     31.34 GiB
Available: 7.53 GiB
Used:      23.81 GiB
Percent:   76.0%
